This notebook really only calls the evaluate_correctness in the evaluation module. It doesn't do anything else as far as I can see. It doesn't evaluate on a large scale.

In [3]:
%load_ext autoreload
%autoreload 2

In [21]:
from langchain.callbacks.tracers import ConsoleCallbackHandler

from meeplemate.evaluation import (
    build_test_case_correctness_evaluation_chain,
    evaluate_correctness,
    preprocess_test_case,
)
from meeplemate.llm_models import load_tgi_chat_model

In [5]:
tgi_url = "http://tgi:80/"

In [6]:
chat_model = load_tgi_chat_model(
    inference_server_url=tgi_url,
    max_new_tokens=512,
    timeout=900,
    do_sample=False,
    temperature=0.01,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [24]:
test_case = {
    "query": "When can I discard a Race card?",
    "true_answers": [
        "You can discard a Race card at any time.",
        "You can discard a Race card at any time, causing you to lose the abilities and bonuses provided by that Race card."
    ],
    "prediction": "You can discard a Race at any time.",
    "wrong_examples": [
        {
            "prediction": "You can discard a Race card at any time, except during combat.",
            "reasoning": "The student answer contradicts the true answers. The true answers specify that a Race card can be discard at any time, but student states that the race card cannot be discarded during combat, contradicting the true answers."
        }
    ]
}

In [25]:
eval_chain = build_test_case_correctness_evaluation_chain(chat_model=chat_model, consistency_samples=2)
eval_chain = eval_chain.with_config({"callbacks": [ConsoleCallbackHandler()]})
result = eval_chain.invoke(preprocess_test_case(test_case))
result

[chain/start] [1:chain:RunnableSequence] Entering Chain run with input:
{
  "query": "When can I discard a Race card?",
  "true_answers": [
    "You can discard a Race card at any time.",
    "You can discard a Race card at any time, causing you to lose the abilities and bonuses provided by that Race card."
  ],
  "prediction": "You can discard a Race at any time.",
  "wrong_examples": [
    {
      "prediction": "You can discard a Race card at any time, except during combat.",
      "reasoning": "The student answer contradicts the true answers. The true answers specify that a Race card can be discard at any time, but student states that the race card cannot be discarded during combat, contradicting the true answers."
    }
  ],
  "examples": [
    {
      "query": "When can I discard a Race card?",
      "true_answers": [
        "You can discard a Race card at any time.",
        "You can discard a Race card at any time, causing you to lose the abilities and bonuses provided by that 

{'score': 1,
 'value': 'CORRECT',
 'reasoning': 'The student answer is the same as True Answer 1, stating that you can discard a Race card at any time. There is no contradiction with the true answers, and the student answer does not provide additional information that conflicts with the true answers.',
 'ratio_correct': 1.0}

In [26]:
evaluate_correctness(
    chat_model.bind(temperature=0.4),
    [test_case],
)

{'results': [{'query': 'When can I discard a Race card?',
   'true_answers': ['You can discard a Race card at any time.',
    'You can discard a Race card at any time, causing you to lose the abilities and bonuses provided by that Race card.'],
   'prediction': 'You can discard a Race at any time.',
   'wrong_examples': [{'prediction': 'You can discard a Race card at any time, except during combat.',
     'reasoning': 'The student answer contradicts the true answers. The true answers specify that a Race card can be discard at any time, but student states that the race card cannot be discarded during combat, contradicting the true answers.'}],
   'examples': [{'query': 'When can I discard a Race card?',
     'true_answers': ['You can discard a Race card at any time.',
      'You can discard a Race card at any time, causing you to lose the abilities and bonuses provided by that Race card.'],
     'prediction': 'You can discard a Race card at any time, except during combat.',
     'wrong_